In [1]:
import pandas as pd #PANDAS
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import statistics
import numpy as np
import sys #GETTING EQUATIONS
import os
sys.path.append('../../bgc-analysis')
from  calc_sensor import pH_sbe, o2_sbe, mcoms
from datetime import datetime
from db_api import get_continuous_one_profile,get_continuous_one_float
from calc_sensor import pH_sbe
sys.path.clear
sys.path.append('C:/Users/athay/Documents/Repositories/test-analysis')
from get_cal_sn import get_cal
sys.path.clear
sys.path.append('C:/Users/athay/Documents/Repositories/flowthrough_testing')
from flowthrough_analysis import diff_analysis

In [2]:
# Input the date 
date = "27-Apr-2026"
platformtype = "&PLATFORM_TYPE=SOLO_BGC_MRV"

In [3]:
folderpath = 'Z:\\www\\bgc\\flow_through\\BGC_SOLO\\'+date
try:
    os.mkdir(folderpath)
except:
    pass


In [4]:
# Read Table
data = pd.read_excel(f"G:/Shared drives/WHOI Argo/BGC/Float Tracking/flow_through_data/{date}.xlsx")
data['fid'] =data['fid'].astype('category')
data.rename(columns={'fid':'Float Id','p':'Pressure (dbar)','t':'Temperature (C)','s':'Salinity (PSU)','no3':'NO3 (uM)','time':'Time'},inplace = True)

In [5]:
for sn in data['Float Id'].unique():
    sns = str(sn)
    cal = get_cal(sn,platformtype)
    sub = data.loc[data['Float Id'] == sn,:]
    #PH
    data.loc[data['Float Id'] == sn, 'PH_TOTAL'] = pH_sbe(cal['TRANSISTOR_PH']['PREDEPLOYMENT_CALIB_COEFFICIENT'],sub['phvrs'], sub['Pressure (dbar)'], sub['Temperature (C)'], sub['Salinity (PSU)'])[0]
    #Oxygen
    data.loc[data['Float Id'] == sn, 'DOXY'] = (o2_sbe(cal['OPTODE_DOXY']['PREDEPLOYMENT_CALIB_COEFFICIENT'], sub['doxyphasedelay'], sub['doxytempvolts'], sub['Pressure (dbar)'], sub['Temperature (C)'], sub['Salinity (PSU)'])[0])


In [6]:
params = ['Pressure (dbar)','Temperature (C)','Salinity (PSU)','doxyphasedelay','doxytempvolts','phvrs','phvk','eco1','eco2','eco3','ocr1','ocr2','ocr3','ocr4','NO3 (uM)','PH_TOTAL','DOXY']

In [7]:
for i in params:
    fig = px.scatter(data,x = data['Time'], y = data[i], color = data['Float Id'])
    fig.write_html(folderpath +'\\' + str(i) + ".html")